# Ejemplo B — Supervisor

Varios **especialistas** y un **orquestador** que decide a quién llamar.

**Qué verás:** state compartido, decisión estructurada, grafo supervisor ⇄ especialistas → synthesize.

**Caso:** el mismo viaje 500€ / 3 noches. El supervisor **no** replanifica: si el vuelo come el budget, sigue igual (estructura más endeble que Plan-and-Execute).


## 1. Preparar el entorno

Node, dependencias + especialistas + grafo supervisor.


In [ ]:
# Setup: Node + dependencias + codigo del ejemplo
import json, subprocess
from pathlib import Path

print("1) Node.js...")
subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/tmp/ns.log 2>&1", shell=True)
subprocess.run("apt-get install -y nodejs >/tmp/apt-node.log 2>&1", shell=True)
print("  ", subprocess.check_output(["node", "-v"], text=True).strip(),
      subprocess.check_output(["npm", "-v"], text=True).strip())

print("2) Dependencias...")
Path("package.json").write_text(json.dumps({
    "name": "workshop-supervisor-colab",
    "private": True,
    "type": "module",
    "dependencies": {
        "@langchain/core": "^1.2.12",
        "@langchain/langgraph": "^1.4.17",
        "@langchain/openai": "^1.5.13",
        "dotenv": "^18.0.3",
        "langchain": "^1.5.14",
        "tsx": "^4.23.15",
        "zod": "^3.24.2",
    },
}, indent=2))
subprocess.run(["npm", "install", "--silent"], check=True)
print("   ok")

print("3) Helpers + ficheros del ejemplo...")
FILES = json.loads('{"lib/catalog.ts": "/**\\n * Catálogo mock BCN ↔ NYC calibrado para 500€ / 3 noches:\\n * no hay solución que cumpla tiempo + calidad/centro + ocio a la vez.\\n * Mover precios aquí (p.ej. directo más barato) cambia el “drama” del taller.\\n * Solo datos — la lógica de búsqueda vive en tools.ts.\\n */\\n\\nexport type Flight = {\\n  id: string;\\n  from: string;\\n  to: string;\\n  price: number;\\n  airline: string;\\n  durationHours: number;\\n  stops: number;\\n};\\n\\nexport type Hotel = {\\n  id: string;\\n  city: string;\\n  name: string;\\n  pricePerNight: number;\\n  stars: number;\\n  neighborhood: string;\\n  distanceKmCenter: number;\\n};\\n\\nexport type Activity = {\\n  id: string;\\n  city: string;\\n  title: string;\\n  price: number;\\n  durationHours: number;\\n  tags: string[];\\n};\\n\\nexport const FLIGHTS: Flight[] = [\\n  {\\n    id: \\"FL-BCN-NYC-CHEAP\\",\\n    from: \\"BCN\\",\\n    to: \\"NYC\\",\\n    price: 95,\\n    airline: \\"BudgetAir\\",\\n    durationHours: 20,\\n    stops: 2,\\n  },\\n  {\\n    id: \\"FL-BCN-NYC-MID\\",\\n    from: \\"BCN\\",\\n    to: \\"NYC\\",\\n    price: 165,\\n    airline: \\"Level\\",\\n    durationHours: 13,\\n    stops: 1,\\n  },\\n  {\\n    id: \\"FL-BCN-NYC-DIRECT\\",\\n    from: \\"BCN\\",\\n    to: \\"NYC\\",\\n    price: 215,\\n    airline: \\"Delta\\",\\n    durationHours: 8.5,\\n    stops: 0,\\n  },\\n  {\\n    id: \\"FL-NYC-BCN-CHEAP\\",\\n    from: \\"NYC\\",\\n    to: \\"BCN\\",\\n    price: 105,\\n    airline: \\"BudgetAir\\",\\n    durationHours: 21,\\n    stops: 2,\\n  },\\n  {\\n    id: \\"FL-NYC-BCN-MID\\",\\n    from: \\"NYC\\",\\n    to: \\"BCN\\",\\n    price: 170,\\n    airline: \\"Level\\",\\n    durationHours: 13.5,\\n    stops: 1,\\n  },\\n  {\\n    id: \\"FL-NYC-BCN-DIRECT\\",\\n    from: \\"NYC\\",\\n    to: \\"BCN\\",\\n    price: 225,\\n    airline: \\"Delta\\",\\n    durationHours: 8.2,\\n    stops: 0,\\n  },\\n];\\n\\nexport const HOTELS: Hotel[] = [\\n  {\\n    id: \\"HT-NYC-1\\",\\n    city: \\"NYC\\",\\n    name: \\"Times Budget Pod\\",\\n    pricePerNight: 45,\\n    stars: 1,\\n    neighborhood: \\"Midtown\\",\\n    distanceKmCenter: 0.8,\\n  },\\n  {\\n    id: \\"HT-NYC-2\\",\\n    city: \\"NYC\\",\\n    name: \\"Queens Transit Inn\\",\\n    pricePerNight: 55,\\n    stars: 5,\\n    neighborhood: \\"Jamaica (Queens)\\",\\n    distanceKmCenter: 14,\\n  },\\n  {\\n    id: \\"HT-NYC-3\\",\\n    city: \\"NYC\\",\\n    name: \\"Brooklyn Mid Stay\\",\\n    pricePerNight: 78,\\n    stars: 3,\\n    neighborhood: \\"Bushwick\\",\\n    distanceKmCenter: 6,\\n  },\\n  {\\n    id: \\"HT-NYC-4\\",\\n    city: \\"NYC\\",\\n    name: \\"SoHo Loft Hotel\\",\\n    pricePerNight: 125,\\n    stars: 4,\\n    neighborhood: \\"SoHo\\",\\n    distanceKmCenter: 1.2,\\n  },\\n];\\n\\nexport const ACTIVITIES: Activity[] = [\\n  {\\n    id: \\"AC-NYC-1\\",\\n    city: \\"NYC\\",\\n    title: \\"Central Park walk\\",\\n    price: 0,\\n    durationHours: 2,\\n    tags: [\\"free\\", \\"outdoors\\"],\\n  },\\n  {\\n    id: \\"AC-NYC-2\\",\\n    city: \\"NYC\\",\\n    title: \\"Brooklyn Bridge + DUMBO\\",\\n    price: 0,\\n    durationHours: 2.5,\\n    tags: [\\"free\\", \\"views\\"],\\n  },\\n  {\\n    id: \\"AC-NYC-3\\",\\n    city: \\"NYC\\",\\n    title: \\"MoMA entrada\\",\\n    price: 30,\\n    durationHours: 3,\\n    tags: [\\"museum\\"],\\n  },\\n  {\\n    id: \\"AC-NYC-4\\",\\n    city: \\"NYC\\",\\n    title: \\"Summit One Vanderbilt\\",\\n    price: 42,\\n    durationHours: 1.5,\\n    tags: [\\"views\\"],\\n  },\\n  {\\n    id: \\"AC-NYC-5\\",\\n    city: \\"NYC\\",\\n    title: \\"Food tour Lower East Side\\",\\n    price: 65,\\n    durationHours: 3.5,\\n    tags: [\\"food\\"],\\n  },\\n  {\\n    id: \\"AC-NYC-6\\",\\n    city: \\"NYC\\",\\n    title: \\"Broadway show (categoría barata)\\",\\n    price: 89,\\n    durationHours: 2.5,\\n    tags: [\\"show\\"],\\n  },\\n  {\\n    id: \\"AC-NYC-7\\",\\n    city: \\"NYC\\",\\n    title: \\"Statue of Liberty + Ellis Island\\",\\n    price: 48,\\n    durationHours: 4,\\n    tags: [\\"iconic\\"],\\n  },\\n];\\n", "lib/types.ts": "/** Tipos / schemas de picks — no es el state del grafo. */\\n\\nimport { z } from \\"zod\\";\\n\\nexport const FlightPickSchema = z.object({\\n  id: z.string().describe(\\"id de ida o compuesto ida+vuelta\\"),\\n  airline: z.string(),\\n  from: z.string(),\\n  to: z.string(),\\n  price: z.number().describe(\\"SUMA real outboundPrice + returnPrice\\"),\\n  outboundPrice: z.number(),\\n  returnPrice: z.number(),\\n  outboundId: z.string(),\\n  returnId: z.string(),\\n  stops: z.number().describe(\\"Máximo de escalas entre ida y vuelta\\"),\\n  durationHours: z.number().describe(\\"Suma de horas ida + vuelta\\"),\\n  reason: z.string().describe(\\"Qué priorizaste y qué cediste\\"),\\n});\\n\\nexport const HotelPickSchema = z.object({\\n  id: z.string(),\\n  name: z.string(),\\n  pricePerNight: z.number().describe(\\"€/noche REAL del catálogo\\"),\\n  stars: z.number(),\\n  neighborhood: z.string(),\\n  distanceKmCenter: z.number().describe(\\"km al centro\\"),\\n  reason: z.string().describe(\\"Qué priorizaste y qué cediste\\"),\\n});\\n\\nexport const ActivityPickSchema = z.object({\\n  id: z.string(),\\n  title: z.string(),\\n  price: z.number(),\\n  durationHours: z.number(),\\n  reason: z.string(),\\n});\\n\\nexport const LeisureOutSchema = z.object({\\n  activities: z.array(ActivityPickSchema),\\n});\\n\\nexport type FlightPick = z.infer<typeof FlightPickSchema>;\\nexport type HotelPick = z.infer<typeof HotelPickSchema>;\\nexport type ActivityPick = z.infer<typeof ActivityPickSchema>;\\n\\nexport type TripContext = {\\n  userRequest: string;\\n  destination: string;\\n  origin: string;\\n  nights: number;\\n  /** Presupuesto TOTAL del viaje (global). */\\n  budget: number;\\n  /**\\n   * Tope de ESTA llamada al especialista (lo decide el orquestador, no el agente).\\n   * 0 = policy de dominio sin €; >0 = orientación de precio.\\n   * Si vuelos siempre van con >0 desde el principio, el arco “directo caro → hotel imposible” desaparece.\\n   */\\n  callBudget?: number;\\n  flight: FlightPick | null;\\n  hotel: HotelPick | null;\\n  activities: ActivityPick[];\\n  /** Solo P&E lo rellena en replan; el supervisor no usa guidance. */\\n  guidance?: string;\\n};\\n", "lib/budget.ts": "/** Aritmética de presupuesto fuera del LLM (los modelos fallan con restas/divisiones). */\\n\\nimport type { TripContext } from \\"./types.ts\\";\\n\\ntype BudgetSlice = {\\n  budget: number;\\n  nights: number;\\n  flight: TripContext[\\"flight\\"];\\n  hotel: TripContext[\\"hotel\\"];\\n  activities: TripContext[\\"activities\\"];\\n};\\n\\nexport function spent(ctx: BudgetSlice): number {\\n  const hotelTotal =\\n    (ctx.hotel?.pricePerNight ?? 0) * (ctx.nights || 0);\\n  const acts = (ctx.activities ?? []).reduce((s, a) => s + a.price, 0);\\n  return (ctx.flight?.price ?? 0) + hotelTotal + acts;\\n}\\n\\n/** Restante del viaje. `null` = sin tope (budget 0 / no informado). */\\nexport function remaining(ctx: BudgetSlice): number | null {\\n  if (!(ctx.budget > 0)) return null;\\n  return ctx.budget - spent(ctx);\\n}\\n", "lib/trace.ts": "/** Trazas de consola (colores + bloques legibles). */\\n\\nimport type { Flight, Hotel, Activity } from \\"./catalog.ts\\";\\nimport type { FlightPick, HotelPick, ActivityPick } from \\"./types.ts\\";\\n\\nexport const c = {\\n  dim: \\"\\\\x1b[2m\\",\\n  bold: \\"\\\\x1b[1m\\",\\n  cyan: \\"\\\\x1b[36m\\",\\n  green: \\"\\\\x1b[32m\\",\\n  yellow: \\"\\\\x1b[33m\\",\\n  magenta: \\"\\\\x1b[35m\\",\\n  blue: \\"\\\\x1b[34m\\",\\n  red: \\"\\\\x1b[31m\\",\\n  reset: \\"\\\\x1b[0m\\",\\n} as const;\\n\\nconst rule = (ch = \\"─\\", n = 60) => `${c.dim}${ch.repeat(n)}${c.reset}`;\\n\\nexport function logSection(icon: string, title: string) {\\n  console.log(`\\\\n${rule()}`);\\n  console.log(`${c.bold}${icon}  ${title}${c.reset}`);\\n  console.log(rule());\\n}\\n\\nexport function logRequest(text: string) {\\n  logSection(\\"🗣️\\", \\"REQUEST\\");\\n  console.log(`   ${text}`);\\n}\\n\\nexport function logResponse(text: string, meta?: string) {\\n  logSection(\\"💬\\", \\"RESPUESTA\\");\\n  for (const line of text.split(\\"\\\\n\\")) {\\n    console.log(`   ${line}`);\\n  }\\n  if (meta) console.log(`\\\\n   ${c.dim}${meta}${c.reset}`);\\n}\\n\\nexport function logFinal(text: string) {\\n  logSection(\\"✨\\", \\"TU PROPUESTA DE VIAJE\\");\\n  for (const line of text.split(\\"\\\\n\\")) {\\n    console.log(`   ${line}`);\\n  }\\n}\\n\\n/** Mensaje final al usuario: oferta comercial, no dump de state. */\\nexport function logOffer(text: string) {\\n  console.log(`\\\\n${rule(\\"═\\")}`);\\n  console.log(`${c.bold}${c.magenta}  ✈  OFERTA LISTA PARA TI${c.reset}`);\\n  console.log(rule(\\"═\\"));\\n  console.log(\\"\\");\\n  for (const line of text.split(\\"\\\\n\\")) {\\n    console.log(`  ${line}`);\\n  }\\n  console.log(\\"\\");\\n  console.log(rule(\\"═\\"));\\n}\\n\\nexport function logEvent(icon: string, title: string, detail?: string) {\\n  console.log(\\n    `\\\\n${c.bold}${icon}  ${title}${c.reset}` +\\n      (detail ? `  ${c.dim}${detail}${c.reset}` : \\"\\")\\n  );\\n}\\n\\nexport function logKv(\\n  rows: Array<[string, string | number | boolean | null | undefined]>\\n) {\\n  for (const [k, v] of rows) {\\n    const shown =\\n      v === null || v === undefined || v === \\"\\"\\n        ? `${c.dim}—${c.reset}`\\n        : `${c.cyan}${String(v)}${c.reset}`;\\n    console.log(`   ${c.dim}│${c.reset}  ${k.padEnd(16)} ${shown}`);\\n  }\\n}\\n\\nexport function logPick(icon: string, title: string, lines: string[]) {\\n  console.log(`\\\\n${c.bold}${icon}  ${title}${c.reset}`);\\n  for (const line of lines) {\\n    console.log(`   ${c.dim}│${c.reset}  ${line}`);\\n  }\\n}\\n\\nexport function traceFlightPick(flight: FlightPick) {\\n  logPick(\\"✈️\\", \\"Vuelo elegido (ida + vuelta)\\", [\\n    `${c.cyan}${flight.from}${c.reset} → ${c.cyan}${flight.to}${c.reset}   ${c.bold}${flight.airline}${c.reset}`,\\n    `Total ${c.bold}${c.green}${flight.price}€${c.reset}  ·  escalas ~${flight.stops}  ·  ~${flight.durationHours}h`,\\n    `${c.dim}${flight.reason}${c.reset}`,\\n  ]);\\n}\\n\\nexport function traceHotelPick(hotel: HotelPick | null, nights: number) {\\n  if (!hotel?.id) {\\n    logPick(\\"🏨\\", \\"Hotel elegido\\", [`${c.red}sin hotel${c.reset}`]);\\n    return;\\n  }\\n  logPick(\\"🏨\\", \\"Hotel elegido\\", [\\n    `${c.bold}${hotel.name}${c.reset}  ·  ${\\"★\\".repeat(hotel.stars)}  ·  ${hotel.neighborhood}`,\\n    `${hotel.pricePerNight}€/noche × ${nights} = ${c.green}${hotel.pricePerNight * nights}€${c.reset}`,\\n    `${c.dim}${hotel.reason}${c.reset}`,\\n  ]);\\n}\\n\\nexport function traceLeisurePick(activities: ActivityPick[]) {\\n  logPick(\\n    \\"🎭\\",\\n    `Ocio · ${activities.length} act.`,\\n    activities.length\\n      ? activities.map(\\n          (a) =>\\n            `${c.bold}${a.title}${c.reset}  ${c.green}${a.price}€${c.reset}  ${a.durationHours}h  ${c.dim}${a.reason}${c.reset}`\\n        )\\n      : [`${c.dim}(ninguna)${c.reset}`]\\n  );\\n}\\n\\nfunction toolHeader(icon: string, title: string, count: number) {\\n  console.log(\\n    `\\\\n${c.bold}${icon}  ${title}${c.reset}  ${c.dim}(${count} resultado${count === 1 ? \\"\\" : \\"s\\"})${c.reset}`\\n  );\\n}\\n\\nfunction toolRow(line: string) {\\n  console.log(`   ${c.dim}│${c.reset}  ${line}`);\\n}\\n\\nexport function traceFlights(\\n  origin: string,\\n  destination: string,\\n  options: Flight[]\\n) {\\n  toolHeader(\\"🛫\\", `Vuelos  ${origin}  →  ${destination}`, options.length);\\n  if (options.length === 0) {\\n    toolRow(`${c.dim}(sin resultados)${c.reset}`);\\n    return;\\n  }\\n  for (const f of options) {\\n    const stopsLabel =\\n      f.stops === 0\\n        ? `${c.green}directo${c.reset}`\\n        : `${c.yellow}${f.stops} escala${f.stops > 1 ? \\"s\\" : \\"\\"}${c.reset}`;\\n    toolRow(\\n      `${c.cyan}${f.from}${c.reset} → ${c.cyan}${f.to}${c.reset}` +\\n        `   ${c.green}${f.price}€${c.reset}` +\\n        `   ${c.yellow}${f.durationHours}h${c.reset}` +\\n        `   ${stopsLabel}` +\\n        `   ${c.bold}${f.airline}${c.reset}` +\\n        `   ${c.dim}${f.id}${c.reset}`\\n    );\\n  }\\n}\\n\\nexport function traceHotels(\\n  destination: string,\\n  maxPricePerNight: number,\\n  options: Hotel[]\\n) {\\n  toolHeader(\\n    \\"🏨\\",\\n    `Hoteles  ${destination}  ·  tope ${maxPricePerNight}€/noche`,\\n    options.length\\n  );\\n  if (options.length === 0) {\\n    toolRow(`${c.dim}(sin resultados)${c.reset}`);\\n    return;\\n  }\\n  for (const h of options) {\\n    toolRow(\\n      `${c.bold}${h.name}${c.reset}` +\\n        `   ${c.green}${h.pricePerNight}€/noche${c.reset}` +\\n        `   ${c.yellow}${\\"★\\".repeat(h.stars)}${c.reset}` +\\n        `   ${c.magenta}${h.neighborhood}${c.reset}` +\\n        `   ${c.cyan}${h.distanceKmCenter} km centro${c.reset}` +\\n        `   ${c.dim}${h.id}${c.reset}`\\n    );\\n  }\\n}\\n\\nexport function traceActivities(destination: string, options: Activity[]) {\\n  toolHeader(\\"🎭\\", `Ocio  ${destination}`, options.length);\\n  if (options.length === 0) {\\n    toolRow(`${c.dim}(sin resultados)${c.reset}`);\\n    return;\\n  }\\n  for (const a of options) {\\n    toolRow(\\n      `${c.bold}${a.title}${c.reset}` +\\n        `   ${c.green}${a.price}€${c.reset}` +\\n        `   ${c.cyan}${a.durationHours}h${c.reset}` +\\n        `   ${c.dim}${a.tags.join(\\" · \\")}${c.reset}` +\\n        `   ${c.dim}${a.id}${c.reset}`\\n    );\\n  }\\n}\\n\\nexport function logBudget(total: number, budget: number) {\\n  const ok = total <= budget;\\n  const color = ok ? c.green : c.red;\\n  const delta = total - budget;\\n  console.log(\\n    `\\\\n   ${c.bold}💶  Total${c.reset}  ${color}${total}€${c.reset}` +\\n      `  ${c.dim}/${c.reset}  budget ${c.cyan}${budget}€${c.reset}` +\\n      `  ${ok ? `${c.green}✓ dentro${c.reset}` : `${c.red}✗ fuera (${delta > 0 ? \\"+\\" : \\"\\"}${delta}€)${c.reset}`}`\\n  );\\n}\\n\\nexport function logTradeoffs(items: string[]) {\\n  if (!items.length) return;\\n  logSection(\\"⚖️\\", \\"CESIONES / TENSIONES\\");\\n  for (const item of items) {\\n    console.log(`   ${c.yellow}•${c.reset}  ${item}`);\\n  }\\n}\\n\\nexport async function printGraphMermaid(\\n  label: string,\\n  source: string | { drawMermaid: () => Promise<string> | string }\\n) {\\n  logSection(\\"📊\\", `GRAFO · ${label}`);\\n  const mermaid =\\n    typeof source === \\"string\\" ? source : await source.drawMermaid();\\n  for (const line of String(mermaid).trimEnd().split(\\"\\\\n\\")) {\\n    console.log(`   ${c.dim}${line}${c.reset}`);\\n  }\\n}\\n", "lib/demos.ts": "/** Caso del taller. Cambiar budget/noches o el catálogo recalibra todo el arco. */\\nexport const WORKSHOP_DEMO =\\n  \\"Organiza un viaje a Nueva York 3 noches desde BCN, presupuesto 500€ (vuelo ida+vuelta, hotel y ocio)\\";\\n", "lib/model.ts": "import { ChatOpenAI } from \\"@langchain/openai\\";\\n\\nexport const model = new ChatOpenAI({ model: \\"gpt-4o-mini\\", temperature: 0 });\\n", "lib/tools.ts": "/**\\n * Tools de viaje: lógica de búsqueda + datos al trace.\\n * El formateo de consola vive en trace.ts.\\n */\\nimport { tool } from \\"langchain\\";\\nimport { z } from \\"zod\\";\\nimport { FLIGHTS, HOTELS, ACTIVITIES } from \\"./catalog.ts\\";\\nimport { traceFlights, traceHotels, traceActivities } from \\"./trace.ts\\";\\n\\nconst CITY_ALIASES: Record<string, string> = {\\n  \\"nueva york\\": \\"NYC\\",\\n  \\"new york\\": \\"NYC\\",\\n  nyc: \\"NYC\\",\\n  jfk: \\"NYC\\",\\n  ewr: \\"NYC\\",\\n  barcelona: \\"BCN\\",\\n  bcn: \\"BCN\\",\\n};\\n\\nfunction city(text: string): string {\\n  const key = text.trim().toLowerCase();\\n  return CITY_ALIASES[key] ?? text.trim().toUpperCase().slice(0, 3);\\n}\\n\\nexport const searchFlightsTool = tool(\\n  ({ destination, origin }: { destination: string; origin: string }) => {\\n    const to = city(destination);\\n    const from = city(origin);\\n    const options = FLIGHTS.filter((f) => f.to === to && f.from === from);\\n    traceFlights(from, to, options);\\n    return JSON.stringify(options);\\n  },\\n  {\\n    name: \\"search_flights\\",\\n    description:\\n      \\"Busca vuelos mock entre origin y destination. Cada tramo incluye price, durationHours, stops, airline.\\",\\n    schema: z.object({\\n      destination: z\\n        .string()\\n        .describe(\\"Ciudad o código destino, ej. Nueva York / NYC\\"),\\n      origin: z.string().describe(\\"Ciudad o código origen, ej. BCN\\"),\\n    }),\\n  }\\n);\\n\\nexport const searchHotelsTool = tool(\\n  ({\\n    destination,\\n    maxPricePerNight,\\n  }: {\\n    destination: string;\\n    maxPricePerNight: number;\\n  }) => {\\n    const dest = city(destination);\\n    const options = HOTELS.filter(\\n      (h) => h.city === dest && h.pricePerNight <= maxPricePerNight\\n    );\\n    traceHotels(dest, maxPricePerNight, options);\\n    return JSON.stringify(options);\\n  },\\n  {\\n    name: \\"search_hotels\\",\\n    description:\\n      \\"Busca hoteles mock. Incluye stars, distanceKmCenter, neighborhood, pricePerNight.\\",\\n    schema: z.object({\\n      destination: z.string().describe(\\"Ciudad o código, ej. NYC\\"),\\n      maxPricePerNight: z\\n        .number()\\n        .describe(\\"Precio máximo por noche en EUR (ej. 200)\\"),\\n    }),\\n  }\\n);\\n\\nexport const searchActivitiesTool = tool(\\n  ({ destination }: { destination: string }) => {\\n    const dest = city(destination);\\n    const options = ACTIVITIES.filter((a) => a.city === dest);\\n    traceActivities(dest, options);\\n    return JSON.stringify(options);\\n  },\\n  {\\n    name: \\"search_activities\\",\\n    description:\\n      \\"Busca actividades mock. Incluye durationHours y price. Cada tramo mañana/tarde aguanta ~3h; >3h ocupa el día entero.\\",\\n    schema: z.object({\\n      destination: z.string().describe(\\"Ciudad o código, ej. NYC\\"),\\n    }),\\n  }\\n);\\n\\nexport const allTravelTools = [\\n  searchFlightsTool,\\n  searchHotelsTool,\\n  searchActivitiesTool,\\n];\\n", "lib/agentes/flights.ts": "import { createAgent } from \\"langchain\\";\\nimport { model } from \\"../model.ts\\";\\nimport { searchFlightsTool } from \\"../tools.ts\\";\\nimport { TripContext, FlightPick, FlightPickSchema } from \\"../types.ts\\";\\nimport { traceFlightPick } from \\"../trace.ts\\";\\n\\n// Policy 0 vs >0 + few-shots: aquí se decide si el demo elige Delta 440€ o Level 335€.\\nconst SYSTEM_PROMPT = `Eres el especialista de VUELOS de una agencia.\\nTu único trabajo: elegir un vuelo de IDA Y VUELTA (round-trip) entre origen y destino.\\nNo inventes vuelos, precios ni IDs: solo datos de search_flights.\\n\\n# Procedimiento\\n1. Llama search_flights para la IDA (origin → destination).\\n2. Llama search_flights para la VUELTA (destination → origin).\\n3. Elige UN tramo de cada lista siguiendo las políticas de selección listadas a continuación.\\n\\n# Políticas de selección\\nLee el presupuesto del mensaje (y \\"Hint:\\" si viene).\\n\\n- Default / presupuesto = 0:\\n  Prioriza SIEMPRE tiempo: menos escalas y menos durationHours.\\n  El precio es secundario. Elige el round-trip más rápido/cómodo.\\n\\n- presupuesto > 0:\\n  Es una orientación de precio total ida+vuelta (~X€), no un techo legal duro.\\n  Prioriza PRECIO: elige el round-trip que mejor se ciña a ~X€.\\n  Puedes pasarte un poco si no hay opción exacta: coge la más cercana posible a X\\n  (si hay empate cerca de X, queda el más rápido entre esos).\\n  Explica el desvío en reason.\\n\\n- Si hay \\"Hint:\\": esa instrucción manda sobre el default (p. ej. un replan que pide precio).\\n\\n# Few-shots (ilustrativos; los IDs/precios reales salen de la tool)\\n\\nEjemplo A — presupuesto=0 (mejor por tiempo)\\nOpciones ida: A 95€ 2 escalas 20h | B 165€ 1 escala 13h | C 215€ 0 escalas 8.5h\\nOpciones vuelta: A\' 105€ 2esc 21h | B\' 170€ 1esc 13.5h | C\' 225€ 0esc 8.2h\\n→ Elige C+C\' (mismo carrier, directo). price=440. reason: \\"Prioricé tiempo: directos ida y vuelta.\\"\\n\\nEjemplo B — presupuesto=335 (ciñete al precio)\\nMismas opciones.\\n→ Elige B+B\'. price=335. reason: \\"Me ajusté a ~335€; acepté 1 escala por tramo.\\"\\n\\nEjemplo C — presupuesto=300 (no hay exacto; el más cercano)\\nMismas opciones (200 / 335 / 440).\\n→ Elige B+B\' (335): más cerca de 300 que 200 (−100) o 440 (+140).\\n  reason: \\"No había ~300€ exactos; 335€ era lo más cercano.\\"\\n`;\\n\\nconst flightsAgent = createAgent({\\n  model,\\n  tools: [searchFlightsTool],\\n  systemPrompt: SYSTEM_PROMPT,\\n  responseFormat: FlightPickSchema, // el schema ya describe price=suma, etc.\\n});\\n\\nexport async function runFlightsSpecialist(\\n  ctx: TripContext\\n): Promise<{ flight: FlightPick }> {\\n  const {\\n    origin,\\n    destination,\\n    callBudget = 0,\\n    guidance,\\n  } = ctx;\\n  // Al agente: \\"presupuesto\\", no el nombre interno callBudget.\\n  const hint = guidance?.trim() ? `\\\\nHint: ${guidance}` : \\"\\";\\n  const result = await flightsAgent.invoke({\\n    messages: [\\n      {\\n        role: \\"user\\",\\n        content: `Origen=${origin}, Destino=${destination}. Presupuesto=${callBudget}.${hint}`,\\n      },\\n    ],\\n  });\\n  const flight = result.structuredResponse as FlightPick;\\n  traceFlightPick(flight);\\n  return { flight };\\n}\\n", "lib/agentes/hotels.ts": "import { createAgent } from \\"langchain\\";\\nimport { model } from \\"../model.ts\\";\\nimport { searchHotelsTool } from \\"../tools.ts\\";\\nimport { TripContext, HotelPick, HotelPickSchema } from \\"../types.ts\\";\\nimport { traceHotelPick } from \\"../trace.ts\\";\\n\\n// El mensaje lleva €/noche ya calculados; si el catálogo no cabe, un salto a 999 (no +1€).\\nconst SYSTEM_PROMPT = `Eres el especialista de HOTELES de una agencia.\\nTu único trabajo: elegir UN hotel en el destino para N noches.\\nNo inventes hoteles, precios ni IDs: solo datos de search_hotels.\\n\\n# Procedimiento\\n1. Llama search_hotels con maxPricePerNight = presupuesto (si presupuesto=0, usa 999).\\n2. Si la lista sale VACÍA: UNA sola búsqueda más con maxPricePerNight=999 (sin tope).\\n   NO subas de 1 en 1 ni hagas muchas búsquedas intermedias.\\n3. Elige UN hotel siguiendo las políticas de selección listadas a continuación.\\n\\n# Políticas de selección\\nLee el presupuesto del mensaje: es €/NOCHE (no el total del hotel). También \\"Hint:\\" si viene.\\nEl catálogo cruza calidad y ubicación: puede haber 5★ lejos y 1★ céntrico.\\n\\n- Default / presupuesto = 0:\\n  PRIORIZA calidad + cercanía al centro (estrellas y pocos km). El precio es secundario.\\n\\n- presupuesto > 0 (~X€/noche) y hay hoteles ≤ X:\\n  Elige entre los que caben; equilibra estrellas y distancia al centro.\\n\\n- presupuesto > 0 y la primera búsqueda salió vacía (ya buscaste con 999):\\n  Elige el hotel con pricePerNight MÁS CERCANO a X (puede pasarse). Diló en reason.\\n\\n- Si hay \\"Hint:\\": esa instrucción manda sobre el default.\\n\\n# Few-shots (ilustrativos; los datos reales salen de la tool)\\n\\nOpciones típicas NYC:\\n  Pod 45€/n 1★ 0.8km | Queens 55€/n 5★ 14km | Brooklyn 78€/n 3★ 6km | SoHo 125€/n 4★ 1.2km\\n\\nEjemplo A — presupuesto=0\\n→ SoHo 125€/n. reason: \\"Sin tope: prioricé 4★ cerca del centro.\\"\\n\\nEjemplo B — presupuesto=55\\n→ Queens 55€/n. reason: \\"Cabe en 55€/n; prioricé 5★ aunque está lejos del centro.\\"\\n\\nEjemplo C — presupuesto=20; primera búsqueda vacía → búsqueda sin tope\\n→ Pod 45€/n. reason: \\"Nada ≤20€/n; elegí Pod (45€), lo más cercano al presupuesto.\\"\\n`;\\n\\nconst hotelsAgent = createAgent({\\n  model,\\n  tools: [searchHotelsTool],\\n  systemPrompt: SYSTEM_PROMPT,\\n  responseFormat: HotelPickSchema,\\n});\\n\\nexport async function runHotelsSpecialist(\\n  ctx: TripContext\\n): Promise<{ hotel: HotelPick | null }> {\\n  const {\\n    destination,\\n    nights,\\n    callBudget = 0,\\n    guidance,\\n  } = ctx;\\n  // callBudget llega como € totales del hotel; aquí lo convertimos a €/noche.\\n  const presupuesto =\\n    callBudget > 0 && nights > 0\\n      ? Math.floor(callBudget / nights)\\n      : 0;\\n  const hint = guidance?.trim() ? `\\\\nHint: ${guidance}` : \\"\\";\\n  const result = await hotelsAgent.invoke({\\n    messages: [\\n      {\\n        role: \\"user\\",\\n        content: `Destino=${destination}, noches=${nights}. Presupuesto=${presupuesto}.${hint}`,\\n      },\\n    ],\\n  });\\n  const hotel = (result.structuredResponse as HotelPick) ?? null;\\n  traceHotelPick(hotel?.id ? hotel : null, nights);\\n  return { hotel: hotel?.id ? hotel : null };\\n}\\n", "lib/agentes/leisure.ts": "import { createAgent } from \\"langchain\\";\\nimport { model } from \\"../model.ts\\";\\nimport { searchActivitiesTool } from \\"../tools.ts\\";\\nimport {\\n  TripContext,\\n  ActivityPick,\\n  LeisureOutSchema,\\n} from \\"../types.ts\\";\\nimport { traceLeisurePick } from \\"../trace.ts\\";\\n\\n// callBudget aquí = € totales de ocio (normalmente el remaining tras vuelo+hotel).\\nconst SYSTEM_PROMPT = `Eres el especialista de OCIO de una agencia.\\nTu único trabajo: elegir actividades en el destino.\\nNo inventes actividades, precios ni IDs: solo datos de search_activities.\\n\\n# Procedimiento\\n1. Llama search_activities.\\n2. Elige actividades siguiendo las políticas de selección listadas a continuación.\\n\\n# Políticas de selección\\nLee el presupuesto del mensaje (y \\"Hint:\\" si viene).\\nTiempo: días de ocio ≈ noches − 1; mañana+tarde ~3h; >3h = día entero.\\nreason de cada actividad: mañana/tarde/duración (no \\"slots\\").\\n\\n- Default / presupuesto = 0:\\n  Mejores actividades que quepan en tiempo. El precio es secundario.\\n\\n- presupuesto > 0 (~X€ totales para el ocio):\\n  Que quepan en € (suma ≤ presupuesto) y en tiempo.\\n  Si no cabe nada de pago: free / más baratas.\\n\\n- Si hay \\"Hint:\\": esa instrucción manda sobre el default.\\n`;\\n\\nconst leisureAgent = createAgent({\\n  model,\\n  tools: [searchActivitiesTool],\\n  systemPrompt: SYSTEM_PROMPT,\\n  responseFormat: LeisureOutSchema,\\n});\\n\\nexport async function runLeisureSpecialist(\\n  ctx: TripContext\\n): Promise<{ activities: ActivityPick[] }> {\\n  const {\\n    destination,\\n    nights,\\n    callBudget = 0,\\n    guidance,\\n  } = ctx;\\n  const hint = guidance?.trim() ? `\\\\nHint: ${guidance}` : \\"\\";\\n  const result = await leisureAgent.invoke({\\n    messages: [\\n      {\\n        role: \\"user\\",\\n        content: `Destino=${destination}, noches=${nights}. Presupuesto=${callBudget}.${hint}`,\\n      },\\n    ],\\n  });\\n  const activities = (result.structuredResponse?.activities ??\\n    []) as ActivityPick[];\\n  traceLeisurePick(activities);\\n  return { activities };\\n}\\n", "lib/agentes/synthesize.ts": "import { model } from \\"../model.ts\\";\\nimport { TripContext } from \\"../types.ts\\";\\nimport { logBudget, logOffer } from \\"../trace.ts\\";\\n\\n// Copy final: no tools; totales se calculan aquí (no le pedimos al LLM que sume).\\nconst OFFER_PROMPT = `Eres copywriter de una agencia de viajes. Redactas la OFERTA FINAL para el cliente.\\nNo eres un sistema técnico: no hables de state, JSON, slots, agentes ni cesiones frías.\\n\\nTono: comercial, cálido, claro, en español. Como un email/WhatsApp premium de la agencia.\\nUsa emojis con gusto (✈️ 🏨 🎭 📅 💶 ✅ ⚠️) en títulos y bullets.\\nProhibido: JSON, markdown de plantilla vacía, tablas técnicas, IDs de catálogo (FL-…, HT-…).\\n\\nEstructura sugerida (adapta con naturalidad):\\n\\n🎉 ¡Tu viaje a {destino} está listo!\\nUna frase de enganche (noches, desde dónde, vibe).\\n\\n✈️ Vuelos\\nIda y vuelta en prosa: aerolínea, si es directo o escalas, tiempo aprox, precio total ida+vuelta.\\nSi hubo tradeoff, una frase suave (\\"priorizamos tiempo / ahorramos para el hotel…\\").\\n\\n🏨 Alojamiento\\nNombre, estrellas, barrio, distancia al centro en lenguaje humano, precio/noche y total noches.\\nFrase de por qué encaja.\\n\\n📅 Tu plan día a día\\nDesglosa el viaje por días (usa las noches del JSON):\\n- Día 1: llegada / traslado + qué hacer si cabe\\n- Días intermedios: mañana y tarde con las actividades elegidas repartidas con sentido\\n- Último día: salida / vuelta\\nNo inventes actividades que no estén en el JSON; puedes dejar tramos libres (pasear, hotel…).\\n\\n🎭 Experiencias (resumen)\\nLista corta de lo incluido con duración y precio (o \\"gratis\\").\\n\\n💶 Resumen\\nDesglose breve + total vs presupuesto, con ✅ o aviso si se pasa.\\n\\n💡 Nota de la agencia\\n1–3 bullets honestos de lo que se priorizó o se sacrificó, en tono de asesor.\\n\\nMáx ~35 líneas. Solo datos del JSON de entrada; no inventes vuelos/hoteles/precios/actividades.`;\\n\\nexport async function runSynthesize(\\n  ctx: TripContext\\n): Promise<{ finalItinerary: string }> {\\n  const hotelTotal =\\n    (ctx.hotel?.pricePerNight ?? 0) * (ctx.nights || 0);\\n  const activitiesTotal = (ctx.activities ?? []).reduce(\\n    (s, a) => s + a.price,\\n    0\\n  );\\n  const flightTotal = ctx.flight?.price ?? 0;\\n  const total = flightTotal + hotelTotal + activitiesTotal;\\n\\n  const response = await model.invoke([\\n    {\\n      role: \\"system\\",\\n      content: OFFER_PROMPT,\\n    },\\n    {\\n      role: \\"user\\",\\n      content: JSON.stringify({\\n        request: ctx.userRequest,\\n        destination: ctx.destination,\\n        nights: ctx.nights,\\n        budget: ctx.budget,\\n        flight: ctx.flight,\\n        hotel: ctx.hotel,\\n        hotelTotal,\\n        activities: ctx.activities,\\n        activitiesTotal,\\n        estimatedTotal: total,\\n      }),\\n    },\\n  ]);\\n\\n  const text = response.content.toString().trim();\\n  logOffer(text);\\n  logBudget(total, ctx.budget);\\n  return { finalItinerary: text };\\n}\\n", "lib/agentes/index.ts": "export { runFlightsSpecialist } from \\"./flights.ts\\";\\nexport { runHotelsSpecialist } from \\"./hotels.ts\\";\\nexport { runLeisureSpecialist } from \\"./leisure.ts\\";\\nexport { runSynthesize } from \\"./synthesize.ts\\";\\n", "state.ts": "import { Annotation } from \\"@langchain/langgraph\\";\\nimport {\\n  FlightPick,\\n  HotelPick,\\n  ActivityPick,\\n} from \\"./lib/types.ts\\";\\n\\nexport const SupervisorStateAnnotation = Annotation.Root({\\n  userRequest: Annotation<string>,\\n  destination: Annotation<string>,\\n  origin: Annotation<string>,\\n  nights: Annotation<number>,\\n  budget: Annotation<number>,\\n  next: Annotation<\\n    \\"flights\\" | \\"hotels\\" | \\"leisure\\" | \\"synthesize\\" | \\"finish\\" | \\"\\"\\n  >,\\n  flight: Annotation<FlightPick | null>,\\n  hotel: Annotation<HotelPick | null>,\\n  activities: Annotation<ActivityPick[]>,\\n  finalItinerary: Annotation<string>,\\n});\\n\\nexport type SupervisorState = typeof SupervisorStateAnnotation.State;\\n", "supervisor.ts": "/**\\n * Supervisor: elige next + rellena hechos.\\n * La tabla del prompt lo hace casi determinista; aflojarla = más “multi-agente”, más fallos.\\n */\\nimport { z } from \\"zod\\";\\nimport { SupervisorState } from \\"./state.ts\\";\\nimport { model } from \\"./lib/model.ts\\";\\nimport { logEvent, logKv } from \\"./lib/trace.ts\\";\\n\\nconst SYSTEM_PROMPT = `Eres el SUPERVISOR de un equipo de viaje.\\nDecides el siguiente especialista y preparas hechos (destino, origen, noches, presupuesto total del viaje).\\n\\n# Orden de decisión (mira el Estado; aplica la PRIMERA regla que cuadre)\\n1. Si NO es un viaje → finish (+ rejectionMessage).\\n2. Si ya hay finalItinerary → finish.\\n3. Si hasFlight=false → flights.\\n4. Si hasFlight=true y hasHotel=false → hotels.  (NUNCA leisure sin hotel.)\\n5. Si hasFlight+hasHotel y activitiesCount=0 → leisure\\n   (aunque el presupuesto esté justo o pasado; leisure puede elegir free).\\n6. Si activitiesCount>0 (o no hace falta ocio) → synthesize.\\n7. Nunca finish a medias sin synthesize cuando sí es un viaje.\\n8. No repitas un especialista si su resultado ya está.\\n\\n# Constraints\\n- Presupuesto total del viaje (tú lo rellenas en los hechos).\\n- Pasarse del presupuesto NO aborta: igual leisure → synthesize.\\n- Normaliza origen/destino a códigos cortos (BCN, NYC).\\n\\n# No hagas\\n- NO elijas hotel/vuelo concreto.\\n- NO replanifiques ocio.\\n- NO saltes hotels: sin hotel no hay leisure.\\n`;\\n\\nconst Decision = z.object({\\n  next: z\\n    .enum([\\"flights\\", \\"hotels\\", \\"leisure\\", \\"synthesize\\", \\"finish\\"])\\n    .describe(\\"A quién llamar o si terminar\\"),\\n  reason: z.string(),\\n  destination: z.string(),\\n  origin: z.string(),\\n  nights: z.number().int(),\\n  budget: z.number().describe(\\"Presupuesto TOTAL del viaje en €\\"),\\n  rejectionMessage: z\\n    .string()\\n    .describe(\\"Si next=finish y no es viaje: mensaje. Si no, vacío.\\"),\\n});\\n\\nexport async function supervisorNode(state: SupervisorState) {\\n  const {\\n    userRequest,\\n    destination,\\n    origin,\\n    nights,\\n    budget,\\n    flight,\\n    hotel,\\n    activities = [],\\n    finalItinerary,\\n  } = state;\\n\\n  // Solo flags de progreso: el € por especialista lo ponen los wrappers de run.ts.\\n  const decision = await model.withStructuredOutput(Decision).invoke([\\n    { role: \\"system\\", content: SYSTEM_PROMPT },\\n    {\\n      role: \\"user\\",\\n      content: `Petición: \\"${userRequest}\\"\\n\\nEstado:\\n${JSON.stringify({\\n  destination,\\n  origin,\\n  nights,\\n  presupuesto: budget,\\n  hasFlight: !!flight,\\n  flightPrice: flight?.price ?? null,\\n  hasHotel: !!hotel,\\n  activitiesCount: activities.length,\\n  hasItinerary: Boolean(finalItinerary && finalItinerary.length > 0),\\n})}`,\\n    },\\n  ]);\\n\\n  logEvent(\\"🎛️\\", `Supervisor → ${decision.next}`, decision.reason);\\n  logKv([\\n    [\\"destino\\", decision.destination || undefined],\\n    [\\"origen\\", decision.origin || undefined],\\n    [\\"noches\\", decision.nights || undefined],\\n    [\\"budget\\", decision.budget ? `${decision.budget}€` : undefined],\\n  ]);\\n\\n  const patch: Partial<SupervisorState> = {\\n    next: decision.next,\\n    destination: decision.destination || destination || \\"\\",\\n    origin: decision.origin || origin || \\"BCN\\",\\n    nights: decision.nights || nights || 0,\\n    budget: decision.budget || budget || 0,\\n  };\\n\\n  if (\\n    decision.next === \\"finish\\" &&\\n    !finalItinerary &&\\n    decision.rejectionMessage\\n  ) {\\n    patch.finalItinerary = decision.rejectionMessage;\\n  }\\n\\n  return patch;\\n}\\n", "run.ts": "/**\\n * Ejemplo B — Supervisor / swarm\\n * Grafo: supervisor ⇄ especialistas → synthesize.\\n * Sin evaluate/replan: si el vuelo se come el budget, hotel/ocio aprietan y ya.\\n */\\nimport \\"dotenv/config\\";\\nimport { StateGraph, START, END, MemorySaver } from \\"@langchain/langgraph\\";\\nimport { SupervisorStateAnnotation, SupervisorState } from \\"./state.ts\\";\\nimport { supervisorNode } from \\"./supervisor.ts\\";\\nimport {\\n  runFlightsSpecialist,\\n  runHotelsSpecialist,\\n  runLeisureSpecialist,\\n  runSynthesize,\\n} from \\"./lib/agentes/index.ts\\";\\nimport { TripContext } from \\"./lib/types.ts\\";\\nimport { remaining } from \\"./lib/budget.ts\\";\\nimport { printGraphMermaid } from \\"./lib/trace.ts\\";\\nimport { WORKSHOP_DEMO } from \\"./lib/demos.ts\\";\\nimport { logRequest, c } from \\"./lib/trace.ts\\";\\n\\n// callBudget no vive en el state: se calcula al llamar (vuelos 0, resto = remaining).\\nfunction asCtx(state: SupervisorState, callBudget: number): TripContext {\\n  const ctx: TripContext = {\\n    userRequest: state.userRequest,\\n    destination: state.destination,\\n    origin: state.origin,\\n    nights: state.nights,\\n    budget: state.budget,\\n    callBudget,\\n    flight: state.flight,\\n    hotel: state.hotel,\\n    activities: state.activities ?? [],\\n  };\\n  return ctx;\\n}\\n\\nasync function flightsNode(state: SupervisorState) {\\n  // 0 → prioriza tiempo (Delta caro). Pon remaining() aquí y el drama del demo se aplana.\\n  return runFlightsSpecialist(asCtx(state, 0));\\n}\\nasync function hotelsNode(state: SupervisorState) {\\n  const rem = remaining(state);\\n  return runHotelsSpecialist(asCtx(state, rem == null ? 0 : Math.max(0, rem)));\\n}\\nasync function leisureNode(state: SupervisorState) {\\n  const rem = remaining(state);\\n  return runLeisureSpecialist(asCtx(state, rem == null ? 0 : Math.max(0, rem)));\\n}\\nasync function synthesizeNode(state: SupervisorState) {\\n  return runSynthesize(asCtx(state, 0));\\n}\\n\\nfunction routeSupervisor(state: SupervisorState) {\\n  return state.next || \\"finish\\";\\n}\\n\\nconst graph = new StateGraph(SupervisorStateAnnotation)\\n  .addNode(\\"supervisor\\", supervisorNode)\\n  .addNode(\\"flights\\", flightsNode)\\n  .addNode(\\"hotels\\", hotelsNode)\\n  .addNode(\\"leisure\\", leisureNode)\\n  .addNode(\\"synthesize\\", synthesizeNode)\\n  .addEdge(START, \\"supervisor\\")\\n  .addConditionalEdges(\\"supervisor\\", routeSupervisor, {\\n    flights: \\"flights\\",\\n    hotels: \\"hotels\\",\\n    leisure: \\"leisure\\",\\n    synthesize: \\"synthesize\\",\\n    finish: END,\\n  })\\n  // Tras cada especialista, vuelta al supervisor (a diferencia de P&E, no hay evaluate).\\n  .addEdge(\\"flights\\", \\"supervisor\\")\\n  .addEdge(\\"hotels\\", \\"supervisor\\")\\n  .addEdge(\\"leisure\\", \\"supervisor\\")\\n  .addEdge(\\"synthesize\\", END)\\n  .compile({ checkpointer: new MemorySaver() });\\n\\nawait printGraphMermaid(\\"Supervisor\\", graph.getGraph().drawMermaid());\\n\\nconst empty = {\\n  activities: [],\\n  flight: null,\\n  hotel: null,\\n  finalItinerary: \\"\\",\\n  next: \\"\\" as const,\\n  destination: \\"\\",\\n  origin: \\"\\",\\n  nights: 0,\\n  budget: 0,\\n};\\n\\nlogRequest(WORKSHOP_DEMO);\\nconst out = await graph.invoke(\\n  { ...empty, userRequest: WORKSHOP_DEMO },\\n  {\\n    configurable: { thread_id: \\"supervisor-colab-0\\" },\\n    runName: \\"workshop-supervisor-colab\\",\\n    tags: [\\"workshop\\", \\"supervisor\\", \\"colab\\"],\\n    recursionLimit: 20, // tool loops agresivos pueden comerse el límite\\n  }\\n);\\n\\nvoid out;\\n"}')
for path, content in FILES.items():
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
print("   helpers: catalog, types, budget, trace, demos, model, tools, agentes/")
print("   ejemplo: state.ts + supervisor.ts + run.ts")
print("Listo. Siguiente celda: API keys.")


## 2. API keys

Pega tu `OPENAI_API_KEY`. LangSmith es opcional (trazas en eu.smith).


In [ ]:
# API keys → .env (necesario antes de ejecutar el ejemplo)
from getpass import getpass
from pathlib import Path

openai = getpass("OPENAI_API_KEY: ")
langsmith = getpass("LANGSMITH_API_KEY (Enter para omitir): ") or ""

env = [f"OPENAI_API_KEY={openai}", "LANGCHAIN_CALLBACKS_BACKGROUND=true"]
if langsmith:
    env += [
        "LANGSMITH_TRACING=true",
        "LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com",
        f"LANGSMITH_API_KEY={langsmith}",
        "LANGSMITH_PROJECT=ia-agentic-workshop-colab",
    ]
else:
    env.append("LANGSMITH_TRACING=false")

Path(".env").write_text("\n".join(env) + "\n")
print("OK → .env escrito" + (" (con LangSmith)" if langsmith else " (sin LangSmith)"))


## 3. Idea del patrón

```text
START → supervisor ⇄ (flights | hotels | leisure) → synthesize → END
```

- El **supervisor** cuida constraints globales (budget total, orden).
- Cada **especialista** aplica su policy de dominio.
- **synthesize** redacta la oferta comercial (no elige opciones nuevas).
- `finish` solo si no es viaje o ya hay oferta — no abortar a medias por pasarse del budget.


## 4. El state

```typescript
import { Annotation } from "@langchain/langgraph";
import {
  FlightPick,
  HotelPick,
  ActivityPick,
} from "./lib/types.ts";

export const SupervisorStateAnnotation = Annotation.Root({
  userRequest: Annotation<string>,
  destination: Annotation<string>,
  origin: Annotation<string>,
  nights: Annotation<number>,
  budget: Annotation<number>,
  next: Annotation<
    "flights" | "hotels" | "leisure" | "synthesize" | "finish" | ""
  >,
  flight: Annotation<FlightPick | null>,
  hotel: Annotation<HotelPick | null>,
  activities: Annotation<ActivityPick[]>,
  finalItinerary: Annotation<string>,
});

export type SupervisorState = typeof SupervisorStateAnnotation.State;
```


## 5. Decisión del supervisor (`supervisor.ts`)

Salida Zod: `next` + hechos (destino, origen, noches, budget).

```typescript
const Decision = z.object({
  next: z
    .enum(["flights", "hotels", "leisure", "synthesize", "finish"])
    .describe("A quién llamar o si terminar"),
  reason: z.string(),
  destination: z.string(),
  origin: z.string(),
  nights: z.number().int(),
  budget: z.number().describe("Presupuesto TOTAL del viaje en €"),
  rejectionMessage: z
    .string()
    .describe("Si next=finish y no es viaje: mensaje. Si no, vacío."),
});
```


## 6. Nodo supervisor (`supervisor.ts`)

Lee el state, decide `next`. **No** elige vuelo/hotel concretos.

```typescript
export async function supervisorNode(state: SupervisorState) {
  const {
    userRequest,
    destination,
    origin,
    nights,
    budget,
    flight,
    hotel,
    activities = [],
    finalItinerary,
  } = state;

  // Solo flags de progreso: el € por especialista lo ponen los wrappers de run.ts.
  const decision = await model.withStructuredOutput(Decision).invoke([
    { role: "system", content: SYSTEM_PROMPT },
    {
      role: "user",
      content: `Petición: "${userRequest}"

Estado:
${JSON.stringify({
  destination,
  origin,
  nights,
  presupuesto: budget,
  hasFlight: !!flight,
  flightPrice: flight?.price ?? null,
  hasHotel: !!hotel,
  activitiesCount: activities.length,
  hasItinerary: Boolean(finalItinerary && finalItinerary.length > 0),
})}`,
    },
  ]);

  logEvent("🎛️", `Supervisor → ${decision.next}`, decision.reason);
  logKv([
    ["destino", decision.destination || undefined],
    ["origen", decision.origin || undefined],
    ["noches", decision.nights || undefined],
    ["budget", decision.budget ? `${decision.budget}€` : undefined],
  ]);

  const patch: Partial<SupervisorState> = {
    next: decision.next,
    destination: decision.destination || destination || "",
    origin: decision.origin || origin || "BCN",
    nights: decision.nights || nights || 0,
    budget: decision.budget || budget || 0,
  };

  if (
    decision.next === "finish" &&
    !finalItinerary &&
    decision.rejectionMessage
  ) {
    patch.finalItinerary = decision.rejectionMessage;
  }

  return patch;
}
```


## 7. El grafo (`run.ts`)

Tras cada especialista → vuelve al supervisor. `synthesize` → `END`.

```typescript
const graph = new StateGraph(SupervisorStateAnnotation)
  .addNode("supervisor", supervisorNode)
  .addNode("flights", flightsNode)
  .addNode("hotels", hotelsNode)
  .addNode("leisure", leisureNode)
  .addNode("synthesize", synthesizeNode)
  .addEdge(START, "supervisor")
  .addConditionalEdges("supervisor", routeSupervisor, {
    flights: "flights",
    hotels: "hotels",
    leisure: "leisure",
    synthesize: "synthesize",
    finish: END,
  })
  // Tras cada especialista, vuelta al supervisor (a diferencia de P&E, no hay evaluate).
  .addEdge("flights", "supervisor")
  .addEdge("hotels", "supervisor")
  .addEdge("leisure", "supervisor")
  .addEdge("synthesize", END)
  .compile({ checkpointer: new MemorySaver() });
```


## 8. Especialistas

Cada uno es un `createAgent` con **su** tool y **su** policy en el system prompt. El human solo pasa hechos (y como mucho un `if` de presupuesto).


### Vuelos

```typescript
const flightsAgent = createAgent({
  model,
  tools: [searchFlightsTool],
  systemPrompt: SYSTEM_PROMPT,
  responseFormat: FlightPickSchema, // el schema ya describe price=suma, etc.
});

export async function runFlightsSpecialist(
  ctx: TripContext
): Promise<{ flight: FlightPick }> {
  const {
    origin,
    destination,
    callBudget = 0,
    guidance,
  } = ctx;
  // Al agente: "presupuesto", no el nombre interno callBudget.
  const hint = guidance?.trim() ? `\nHint: ${guidance}` : "";
  const result = await flightsAgent.invoke({
    messages: [
      {
        role: "user",
        content: `Origen=${origin}, Destino=${destination}. Presupuesto=${callBudget}.${hint}`,
      },
    ],
  });
  const flight = result.structuredResponse as FlightPick;
  traceFlightPick(flight);
  return { flight };
}
```


### Hoteles

```typescript
const hotelsAgent = createAgent({
  model,
  tools: [searchHotelsTool],
  systemPrompt: SYSTEM_PROMPT,
  responseFormat: HotelPickSchema,
});

export async function runHotelsSpecialist(
  ctx: TripContext
): Promise<{ hotel: HotelPick | null }> {
  const {
    destination,
    nights,
    callBudget = 0,
    guidance,
  } = ctx;
  // callBudget llega como € totales del hotel; aquí lo convertimos a €/noche.
  const presupuesto =
    callBudget > 0 && nights > 0
      ? Math.floor(callBudget / nights)
      : 0;
  const hint = guidance?.trim() ? `\nHint: ${guidance}` : "";
  const result = await hotelsAgent.invoke({
    messages: [
      {
        role: "user",
        content: `Destino=${destination}, noches=${nights}. Presupuesto=${presupuesto}.${hint}`,
      },
    ],
  });
  const hotel = (result.structuredResponse as HotelPick) ?? null;
  traceHotelPick(hotel?.id ? hotel : null, nights);
  return { hotel: hotel?.id ? hotel : null };
}
```


### Ocio

```typescript
const leisureAgent = createAgent({
  model,
  tools: [searchActivitiesTool],
  systemPrompt: SYSTEM_PROMPT,
  responseFormat: LeisureOutSchema,
});

export async function runLeisureSpecialist(
  ctx: TripContext
): Promise<{ activities: ActivityPick[] }> {
  const {
    destination,
    nights,
    callBudget = 0,
    guidance,
  } = ctx;
  const hint = guidance?.trim() ? `\nHint: ${guidance}` : "";
  const result = await leisureAgent.invoke({
    messages: [
      {
        role: "user",
        content: `Destino=${destination}, noches=${nights}. Presupuesto=${callBudget}.${hint}`,
      },
    ],
  });
  const activities = (result.structuredResponse?.activities ??
    []) as ActivityPick[];
  traceLeisurePick(activities);
  return { activities };
}
```


## 9. Oferta final

`synthesize` monta la oferta (día a día) a partir del state.

```typescript
export async function runSynthesize(
  ctx: TripContext
): Promise<{ finalItinerary: string }> {
  const hotelTotal =
    (ctx.hotel?.pricePerNight ?? 0) * (ctx.nights || 0);
  const activitiesTotal = (ctx.activities ?? []).reduce(
    (s, a) => s + a.price,
    0
  );
  const flightTotal = ctx.flight?.price ?? 0;
  const total = flightTotal + hotelTotal + activitiesTotal;

  const response = await model.invoke([
    {
      role: "system",
      content: OFFER_PROMPT,
    },
    {
      role: "user",
      content: JSON.stringify({
        request: ctx.userRequest,
        destination: ctx.destination,
        nights: ctx.nights,
        budget: ctx.budget,
        flight: ctx.flight,
        hotel: ctx.hotel,
        hotelTotal,
        activities: ctx.activities,
        activitiesTotal,
        estimatedTotal: total,
      }),
    },
  ]);

  const text = response.content.toString().trim();
  logOffer(text);
  logBudget(total, ctx.budget);
  return { finalItinerary: text };
}
```


## 10. Ejecutar

LangSmith: `workshop-supervisor-colab`. Observa el bucle supervisor → especialista.


In [ ]:
!npx tsx run.ts


## Resumen

| Pieza | Rol |
|-------|-----|
| State | Hechos + `flight` / `hotel` / … |
| Supervisor | Elige `next` + rellena hechos |
| Especialistas | Deciden opciones concretas |
| Debilidad | Sin evaluate/replan → a veces se salta pasos o no corrige un vuelo caro |

**Siguiente:** Ejemplo C — Plan-and-Execute (evaluate + replan + `callBudget`).
